# 1 · Preprocesamiento de datos

> **Tipo de ML:** `supervisado`

## 1. Imports

In [1]:

import pandas as pd
import numpy as np
import joblib

from climasafeai.data.make_dataset import load_data
from climasafeai.features.build_features import preprocess_data
from climasafeai.utils.paths import PROCESSED_DATA_DIR, ARTIFACTS_DIR
from climasafeai.utils.paths import RAW_DATA_DIR, FIGURES_DIR
from climasafeai.data.make_dataset import load_data, dataset_calor, cargar_era5_filtrado, cargar_provincias_unificadas, calcular_puntos_provincia  


## 2. Cargar datos crudos

In [2]:
provincias = cargar_provincias_unificadas()
puntos_por_provincia = calcular_puntos_provincia(provincias, col_nombre="NAMEUNIT")
df_eras = cargar_era5_filtrado(puntos_por_provincia)
df_eras.head(3)

--> Cargando y filtrando 126 archivos ERA5...


<xarray.Dataset> Size: 908B
Dimensions:     (valid_time: 3, punto: 3)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 24B 2016-01-01 ... 2016-01-01T02:...
    expver      (valid_time) <U4 48B '0001' '0001' '0001'
    latitude    (punto) float64 24B 43.25 43.75 42.5
    longitude   (punto) float64 24B -8.5 -7.75 -9.0
    provincia   (punto) <U44 528B 'A Coruña' 'A Coruña' 'A Coruña'
    tipo_punto  (punto) <U6 72B 'centro' 'norte' 'sur'
    number      int64 8B 0
Dimensions without coordinates: punto
Data variables:
    t2m         (valid_time, punto) float32 36B 284.2 285.4 ... 285.5 285.7
    d2m         (valid_time, punto) float32 36B 280.5 280.6 ... 280.1 282.0
    sp          (valid_time, punto) float32 36B 9.955e+04 ... 1.013e+05
    u10         (valid_time, punto) float32 36B -0.7338 0.4088 ... -0.9486
    v10         (valid_time, punto) float32 36B 6.08 5.216 7.805 ... 5.451 9.202
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-07-02T11:29 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:

DATA_FILE = RAW_DATA_DIR / "momo_data.csv"
try:
    df_momo = load_data(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Archivo no encontrado: {DATA_FILE}\n"
        "Coloca el CSV en data/raw/ y ajusta DATA_FILE arriba."
    )
df_momo.head(10)


--> Cargando datos desde /home/cacelas/Documentos/anfaia/ClimaSafeAI/data/raw/momo_data.csv...
    Datos cargados. Dimensiones: (7257600, 15)


,ambito,cod_ambito,cod_ine_ambito,nombre_ambito,cod_sexo,nombre_sexo,cod_gedad,nombre_gedad,fecha_defuncion,defunciones_observadas,defunciones_estimadas_base,defunciones_estimadas_base_q01,defunciones_estimadas_base_q99,defunciones_atrib_exc_temp,defunciones_atrib_def_temp
0,ccaa,AN,1.0,Andalucía,1,hombres,+65,edad >= 65,2015-01-01,108.88,83.25,63.0,105.0,0.0,5.50
1,ccaa,AN,1.0,Andalucía,1,hombres,+85,edad >= 85,2015-01-01,35.57,27.39,16.0,40.0,0.0,1.61
2,ccaa,AN,1.0,Andalucía,1,hombres,0-14,edad 0-14,2015-01-01,0.00,0.61,0.0,3.0,0.0,0.00
3,ccaa,AN,1.0,Andalucía,1,hombres,15-44,edad 15-44,2015-01-01,3.05,3.06,0.0,8.0,0.0,0.02
4,ccaa,AN,1.0,Andalucía,1,hombres,45-64,edad 45-64,2015-01-01,29.44,17.47,9.0,28.0,0.0,0.60
5,ccaa,AN,1.0,Andalucía,1,hombres,65-74,edad 65-74,2015-01-01,32.50,19.73,10.0,31.0,0.0,1.23
6,ccaa,AN,1.0,Andalucía,1,hombres,75-84,edad 75-84,2015-01-01,41.69,34.15,21.0,48.0,0.0,2.08
7,ccaa,AN,1.0,Andalucía,1,hombres,all,todos,2015-01-01,142.24,105.76,83.0,130.0,0.0,6.34
8,ccaa,AN,1.0,Andalucía,6,mujeres,+65,edad >= 65,2015-01-01,121.06,91.61,70.0,115.0,0.0,4.99
9,ccaa,AN,1.0,Andalucía,6,mujeres,+85,edad >= 85,2015-01-01,69.10,49.65,34.0,67.0,0.0,2.60


In [4]:
df = dataset_calor(df_momo, df_eras)

## 3. Preprocesar


Indica la columna objetivo (`TARGET_COL`) y el tipo de scaler.


In [5]:

TARGET_COL = 'defunciones_atrib_exc_temp'   # ← ajusta
SCALER_TYPE = 'standard'            # 'standard' | 'minmax'
X_train, X_test, y_train, y_test = preprocess_data(df, target_col=TARGET_COL, scaler_type=SCALER_TYPE)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Balance clases (train):', y_train.value_counts(normalize=True).to_dict())

--> Preprocesando datos (target='defunciones_atrib_exc_temp', scaler='standard', PCA=None)...
    Encoders guardados → encoders.joblib  (['fecha', 'provincia'])
    Muestras de entrenamiento tras submuestreo: 74999
    Muestras de entrenamiento tras submuestreo: 74999
    feature_names.joblib guardado (11 features)
    Scaler guardado → scaler.joblib
    Train: (74999, 11) | Test: (33957, 11)
    Proporción clases (train): {0.0: 0.8110508140108534, 1.55: 0.0004800064000853345, 0.79: 0.0004800064000853345, 1.51: 0.00046667288897185295, 0.62: 0.00046667288897185295, 0.73: 0.00046667288897185295, 1.09: 0.00044000586674488993, 0.63: 0.0004266723556314084, 0.66: 0.0004266723556314084, 2.0: 0.0004266723556314084, 0.51: 0.0004266723556314084, 2.02: 0.0004266723556314084, 0.52: 0.0004133388445179269, 0.71: 0.0004133388445179269, 1.37: 0.0004133388445179269, 0.53: 0.0004133388445179269, 0.65: 0.00040000533340444537, 1.15: 0.00040000533340444537, 1.53: 0.00040000533340444537, 3.02: 0.00040000533

## 4. Guardar datos procesados

In [6]:

import pandas as pd
pd.DataFrame(X_train).to_csv(PROCESSED_DATA_DIR / 'X_train_calor.csv', index=False)
pd.DataFrame(X_test).to_csv(PROCESSED_DATA_DIR / 'X_test_calor.csv', index=False)
pd.Series(y_train).to_csv(PROCESSED_DATA_DIR / 'y_train_calor.csv', index=False)
pd.Series(y_test).to_csv(PROCESSED_DATA_DIR / 'y_test_calor.csv', index=False)

print('Datos guardados en', PROCESSED_DATA_DIR)


Datos guardados en /home/cacelas/Documentos/anfaia/ClimaSafeAI/data/processed
